# ChemBreak Adaptive Jailbreak V4

Google Cloud Notebook Enterprise runner for the frozen ChemHarm 500-task bank. Start with the mock test. Enable live execution only inside the approved research project.

In [ ]:
# EDIT THIS CELL
GOOGLE_CLOUD_PROJECT = "YOUR_GOOGLE_CLOUD_PROJECT_ID"
PHASE = "test"  # test, pilot, or production
LIVE = False       # keep False for the first complete pass
GCS_CHECKPOINT_URI = None  # example: gs://private-bucket/chembreak/cbj4
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Adaptive_Jailbreak_V4"

## 1. Locate or clone the GitHub project

In [ ]:
from pathlib import Path
import os, subprocess, sys

candidate = Path.cwd()
if (candidate / "pyproject.toml").exists() and candidate.name == PROJECT_SUBDIR:
    PROJECT_DIR = candidate
elif (candidate / PROJECT_SUBDIR / "pyproject.toml").exists():
    PROJECT_DIR = candidate / PROJECT_SUBDIR
else:
    checkout = Path("/home/jupyter/chembreak_repo")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(checkout)], check=True)
    PROJECT_DIR = checkout / PROJECT_SUBDIR

assert (PROJECT_DIR / "pyproject.toml").exists(), f"Project not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print(PROJECT_DIR)

## 2. Install the package

The notebook image should already contain a CUDA-compatible PyTorch build. This cell preserves it and installs the remaining dependencies.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 3. Build the signed runtime configuration

In [ ]:
import yaml
from chembreak_v4.config import canonical_config, load_config

phase_counts = {"test": 8, "pilot": 40, "production": 500}
assert PHASE in phase_counts
base = load_config(PROJECT_DIR / "configs" / "config.test.yaml")
runtime = canonical_config(base)
runtime["run"]["phase"] = PHASE
runtime["run"]["dry_run"] = not LIVE
runtime["run"]["live_acknowledgement"] = LIVE
runtime["run"]["gcs_checkpoint_uri"] = GCS_CHECKPOINT_URI
runtime["experiment"]["task_count"] = phase_counts[PHASE]
runtime_path = PROJECT_DIR / "configs" / f"runtime.{PHASE}.yaml"
runtime_path.write_text(yaml.safe_dump(runtime, sort_keys=False), encoding="utf-8")

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["HF_HOME"] = "/home/jupyter/.cache/huggingface"
if LIVE:
    os.environ["CHEMBREAK_ENABLE_LIVE"] = "YES"
else:
    os.environ.pop("CHEMBREAK_ENABLE_LIVE", None)
print(runtime_path)
print(f"phase={PHASE} live={LIVE} tasks={phase_counts[PHASE]}")

## 4. Preflight

In mock mode this validates the package, frozen dataset, subset, schemas, and local environment without contacting any model. In live mode it also checks the four Google Cloud model roles and the Hugging Face repositories. Add `load_targets=True` only when you want to download and sequentially load all three target models before the run.

In [ ]:
from chembreak_v4.preflight import run_preflight
preflight = run_preflight(runtime_path, load_targets=False)
print({"status": preflight["status"], "gpu": preflight["gpu"], "selected_subset": preflight["selected_subset"]})

## 5. Execute or resume

Progress, completed count, and ETA remain visible throughout the run. Re-running this cell with the same signed configuration skips completed episodes.

In [ ]:
from chembreak_v4.runner import run
RUN_DIR = run(runtime_path)
print(RUN_DIR)

## 6. Review the main result tables

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv(RUN_DIR / "release" / "metrics_overall.csv"))
display(pd.read_csv(RUN_DIR / "release" / "asr_by_query_budget.csv"))
display(pd.read_csv(RUN_DIR / "release" / "paired_comparisons.csv"))
failures = pd.read_csv(RUN_DIR / "release" / "failures.csv")
print(f"recorded failures: {len(failures)}")
display(failures.head(20))

## 7. Freeze before moving phases

Do not move from test to pilot, or from pilot to production, until the previous phase is complete and the failure ledger has been reviewed. If you change prompts, thresholds, reward coefficients, model IDs, or generation settings, the signature changes and a fresh run directory is created.